# 07 – Clinical Note Preprocessing & Temporal Alignment

Turns the raw NOTEEVENTS free text into a per-stay, hourly-aligned text grid matching the
24-hour signal window. First-24h notes only; discharge summaries excluded (label leakage +
unreliable timestamps); de-identification placeholders and whitespace cleaned; notes binned
to the same 24 one-hour bins as the signals.

**Run after notebook 06** — uses modelling_cohort_sepsis_mortality.csv and raw NOTEEVENTS.

**Produces:** hourly_notes_24h.csv (used by notebooks 09a, 09b).

MIMIC-III data not included (PhysioNet DUA); see README. Patient note text outputs cleared.

In [ ]:
# --- Setup ---
import os

try:
    from config import DATA_DIR
except ImportError:
    DATA_DIR = os.environ.get("ERP_DATA_DIR", "./data")

def data_path(name):
    return os.path.join(DATA_DIR, name)

import re
import numpy as np
import pandas as pd


In [ ]:
# modelling cohort (sepsis, 24h window, labelled) built in notebook 06
cohort = pd.read_csv(data_path("modelling_cohort_sepsis_mortality.csv"), low_memory=False)
cohort["INTIME"] = pd.to_datetime(cohort["INTIME"], errors="coerce")
print("Modelling cohort:", cohort.shape)

# we align notes for these ICU stays only
cohort_hadm = set(cohort["HADM_ID"])
# map HADM_ID -> INTIME (note CHARTTIME is per-admission; we window from ICU INTIME)
intime_map = cohort.set_index("HADM_ID")["INTIME"].to_dict()

Modelling cohort: (10068, 14)


## 1. Stream NOTEEVENTS and keep only relevant notes
NOTEEVENTS is ~1.1 GB, so it is read in chunks. For each chunk we keep only:
- notes belonging to a cohort HADM_ID,
- notes that are **not** discharge summaries,
- notes with a valid CHARTTIME.

In [ ]:
NOTE_PATH = data_path("NOTEEVENTS.csv.gz")

usecols = ["SUBJECT_ID", "HADM_ID", "CHARTTIME", "CATEGORY", "ISERROR", "TEXT"]
keep_chunks = []
total_rows = 0

for chunk in pd.read_csv(NOTE_PATH, usecols=usecols, chunksize=100_000, low_memory=False):
    total_rows += len(chunk)
    # restrict to cohort admissions
    chunk = chunk[chunk["HADM_ID"].isin(cohort_hadm)].copy()
    if chunk.empty:
        continue
    # drop error notes
    chunk = chunk[chunk["ISERROR"].isna()]
    # drop discharge summaries (leakage + unreliable timestamp)
    chunk = chunk[chunk["CATEGORY"].str.strip().str.lower() != "discharge summary"]
    # need a valid charttime to place the note in time
    chunk = chunk[chunk["CHARTTIME"].notna()]
    keep_chunks.append(chunk)

notes = pd.concat(keep_chunks, ignore_index=True)
print("Scanned rows:", total_rows)
print("Kept notes (cohort, non-discharge, timestamped):", len(notes))

Scanned rows: 2083180
Kept notes (cohort, non-discharge, timestamped): 502411


## 2. Restrict to the first 24 hours and assign an hour bin

In [ ]:
notes["CHARTTIME"] = pd.to_datetime(notes["CHARTTIME"], errors="coerce")
notes["INTIME"] = notes["HADM_ID"].map(intime_map)
notes = notes[notes["INTIME"].notna() & notes["CHARTTIME"].notna()].copy()

# hours since ICU admission
notes["hours_since_intime"] = (
    (notes["CHARTTIME"] - notes["INTIME"]).dt.total_seconds() / 3600.0
)

# keep only first 24h
notes_24h = notes[(notes["hours_since_intime"] >= 0) & (notes["hours_since_intime"] < 24)].copy()
notes_24h["hour_bin"] = notes_24h["hours_since_intime"].astype(int)  # 0..23

print("Notes within first 24h:", len(notes_24h))
print("ICU admissions with >=1 note in first 24h:", notes_24h["HADM_ID"].nunique())

Notes within first 24h: 62325
ICU admissions with >=1 note in first 24h: 9363


## 3. Clean note text
Remove de-identification placeholders (`[**...**]`), collapse whitespace, strip.

In [ ]:
def clean_text(t):
    if not isinstance(t, str):
        return ""
    t = re.sub(r"\[\*\*.*?\*\*\]", " ", t)   # remove [**...**] de-id placeholders
    t = re.sub(r"\s+", " ", t)                     # collapse whitespace/newlines
    return t.strip()

notes_24h["TEXT_CLEAN"] = notes_24h["TEXT"].apply(clean_text)
# drop notes that became empty after cleaning
notes_24h = notes_24h[notes_24h["TEXT_CLEAN"].str.len() > 0].copy()
print("Notes after cleaning:", len(notes_24h))
print("Example cleaned note:\n", notes_24h["TEXT_CLEAN"].iloc[0][:400])

## 4. Aggregate to (HADM_ID, hour_bin): concatenate note text per hour
Multiple notes in the same hour are joined; this gives one text field per stay-hour.

In [ ]:
notes_24h = notes_24h.sort_values(["HADM_ID", "hour_bin", "CHARTTIME"])

hourly_notes = (
    notes_24h
    .groupby(["HADM_ID", "hour_bin"])["TEXT_CLEAN"]
    .apply(lambda s: " ".join(s))
    .reset_index()
)
print("Stay-hour note rows:", len(hourly_notes))
print(hourly_notes.head())

## 5. Reindex onto the full 24-hour grid (empty hours kept as blank)
Every cohort ICU stay gets 24 rows (hours 0-23); hours without a note are empty
strings, mirroring the signal grid so the two modalities share the same time axis.

In [ ]:
all_hadm = cohort["HADM_ID"].unique()
full_index = pd.MultiIndex.from_product([all_hadm, range(24)], names=["HADM_ID", "hour_bin"])

notes_grid = (
    hourly_notes.set_index(["HADM_ID", "hour_bin"])
    .reindex(full_index)
    .reset_index()
)
notes_grid["TEXT_CLEAN"] = notes_grid["TEXT_CLEAN"].fillna("")
notes_grid["has_note"] = (notes_grid["TEXT_CLEAN"].str.len() > 0).astype(int)

print("Grid shape:", notes_grid.shape, "(expected", len(all_hadm)*24, "rows)")
print("Hours with a note: {:.1%}".format(notes_grid["has_note"].mean()))

Grid shape: (241632, 4) (expected 241632 rows)
Hours with a note: 17.1%


## 6. Coverage summary and save

In [ ]:
notes_per_stay = notes_grid.groupby("HADM_ID")["has_note"].sum()
print("ICU stays with >=1 note in first 24h: {} / {} ({:.1%})".format(
    (notes_per_stay > 0).sum(), len(notes_per_stay),
    (notes_per_stay > 0).mean()))
print("Mean note-hours per stay (of 24):", round(notes_per_stay.mean(), 2))
print("Median note-hours per stay:", int(notes_per_stay.median()))

ICU stays with >=1 note in first 24h: 9363 / 10068 (93.0%)
Mean note-hours per stay (of 24): 4.1
Median note-hours per stay: 4


In [ ]:
notes_grid.to_csv(data_path("hourly_notes_24h.csv"), index=False)
print("Saved:", data_path("hourly_notes_24h.csv"))
print("Shape:", notes_grid.shape)